In [20]:
import numpy as np
import matplotlib.pyplot as plt


from easydynamics.sample import BrownianTranslationalDiffusion

from easydynamics.sample import SampleModel
from easydynamics.sample import LorentzianComponent
from easydynamics.sample import DeltaFunctionComponent

from easydynamics.sample import GaussianComponent

from easydynamics.resolution import ResolutionHandler

from easyscience import Parameter

import scipp as sc

import plopp as pp

%matplotlib widget

In [21]:
# Create some fake data
Q=np.linspace(0.1,2,16)
E=np.linspace(-5,5,1001)

temperatures=[50,100,200,300]
diffusion_coefficients=[0.1,0.25,0.5,0.75]
convoluted_signal=np.zeros((len(temperatures),len(Q),len(E)))

for T in range(len(temperatures)):
    model=BrownianTranslationalDiffusion(name="DiffusionModel", diffusion_coefficient=diffusion_coefficients[T])
    HWHM=model.calculate_width(Q)

    QQISF=model.calculate_QISF(Q)
    EISF=model.calculate_EISF(Q)

    resolution=GaussianComponent(name="Resolution", area=1,width=0.1)

    resolution_handler=ResolutionHandler()

    sample_model=[]
    for i in range(len(Q)):
        sample_model.append(SampleModel(name=f"SampleModel_{i}"))

        sample_model[i].add_component(DeltaFunctionComponent(area=EISF[i]+0.2, name="Elastic"))
        sample_model[i].add_component(LorentzianComponent(area=QQISF[i], name="QuasiElastic", width=HWHM[i]) )

        convoluted_signal[T,i,:] = resolution_handler.convolve(E,sample_model[i],resolution)+0.3+0.1*np.random.normal(size=len(E))



Q_scipp=sc.array(dims=['Q'],values=Q, unit='1/angstrom')
E_scipp=sc.array(dims=['energy'],values=E,unit='meV')
intensity_scipp=sc.array(dims=['Temperature','Q','energy'],values=convoluted_signal,variances=0.1*convoluted_signal)

diffusion_data = sc.DataArray(data=intensity_scipp, coords={'Q':Q_scipp,'energy': E_scipp,'Temperature':sc.array(dims=['Temperature'],values=temperatures)})


pp.slicer(diffusion_data.transpose(),coords=['energy','Q'],keep=['Q','energy'])



InteractiveFigure(children=(HBar(), HBar(children=(VBar(children=(Toolbar(children=(ButtonTool(icon='home', la…

In [23]:
pp.slicer(diffusion_data.transpose(),coords=['energy','Q'],keep=['energy'])


InteractiveFigure(children=(HBar(), HBar(children=(VBar(children=(Toolbar(children=(ButtonTool(icon='home', la…

In [ ]:
from easydynamics.job import Job
from easydynamics.experiment import Experiment
from easydynamics.experiment import Data
from easydynamics.analysis import Analysis
from easydynamics.sample import SampleModel
from easydynamics.sample import PolynomialComponent

diffusion_job= Job(name='BrownianDiffusion')


exp=Experiment()
data=Data()
data.append(diffusion_data)

exp.set_data(data)

diffusion_job.set_experiment(exp)


bg=SampleModel('Background')
bg.add_component(PolynomialComponent(coefficients=[1e-3]))
diffusion_job.set_background_model(bg)

resolution=SampleModel()
resolution.add_component(GaussianComponent(name="Resolution", area=1,width=0.1))
diffusion_job.set_resolution_model(resolution)


diffusion_job.generate_empty_analysis_array()

diffusion_job.set_resolution_model_for_all_analyses()




diffusion_model=BrownianTranslationalDiffusion(name="DiffusionModel", diffusion_coefficient=0.3)
diffusion_job.set_theory_for_all_analyses(diffusion_model)

In [ ]:

# Quick check that the resolution model has been copied to all analyses with new parameters
print(diffusion_job.analysis[0][0]._resolution_model.components['Resolution'].area)
print(diffusion_job.analysis[0][0]._resolution_model.components['Resolution'].area.unique_name)
print(diffusion_job.analysis[1][0]._resolution_model.components['Resolution'].area.unique_name)
print(diffusion_job.analysis[1][0]._resolution_model.components['Resolution'].area)


<Parameter 'Resolution area': 1.0000 meV (fixed), bounds=[0.0:inf]>
Parameter_871
Parameter_823
<Parameter 'Resolution area': 1.0000 meV (fixed), bounds=[0.0:inf]>
